# PVP-Dataset + PV100F 50张 Teacher 门槛测试

本 notebook 只运行 GitHub 已提交代码。测试集包含 20 张常规正样本、15 张主要组件距边缘至少32像素的低覆盖困难正样本，以及15张规则强边缘困难负样本。测试完整影像与掩膜引导局部裁剪，不训练 Student。

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('当前 Kaggle notebook 未分配 GPU，请确认 kernel-metadata.json 中 enable_gpu=true')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
repo_dir = Path('/kaggle/working/internvl-pv-distillation')
repo_url = 'https://github.com/Zh0416/internvl-pv-distillation.git'
fixed_commit = '91760d72b0e97c49c50514f25a9bfd5c92ad8600'

if repo_dir.exists():
    subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', fixed_commit], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip()
if actual_commit != fixed_commit:
    raise RuntimeError(f'Git commit不一致: expected={fixed_commit}, actual={actual_commit}')
print('Repository commit:', actual_commit)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(repo_dir / 'requirements.txt')],
    check=True,
)
os.chdir(repo_dir)

In [ ]:
config_path = repo_dir / 'configs/kaggle_pv_hard_crop_distill.yaml'
mounted_dirs = sorted(
    str(path) for path in Path('/kaggle/input').glob('**/*')
    if path.is_dir() and len(path.relative_to('/kaggle/input').parts) <= 4
)
print('Mounted input directories:', json.dumps(mounted_dirs, ensure_ascii=False, indent=2))

subprocess.run(
    [sys.executable, 'scripts/06_diagnose_hard_samples.py', '--config', str(config_path)],
    check=True,
)
diagnostic_path = Path('/kaggle/working/outputs/reports/hard_sample_diagnostics.json')
diagnostic = json.loads(diagnostic_path.read_text(encoding='utf-8'))
print(json.dumps({
    'hard_positive_count': diagnostic['count'],
    'min_border_margin': diagnostic['selection']['min_border_margin'],
    'selected': [
        {
            'filename': row['sample']['filename'],
            'mask_pixels': row['sample']['pv_pixels'],
            'component_pixels': row['target_component']['pixels'],
            'border_distance': row['target_component']['border_distance'],
        }
        for row in diagnostic['selection']['selected']
    ],
}, ensure_ascii=False, indent=2))

In [ ]:
subprocess.run(
    [sys.executable, 'scripts/07_test_hard_crop_teacher.py', '--config', str(config_path)],
    check=True,
)

In [ ]:
report_path = Path('/kaggle/working/outputs/reports/hard_crop_teacher_report.json')
report = json.loads(report_path.read_text(encoding='utf-8'))
summary = {
    'repository_commit': fixed_commit,
    'training_performed': report['training_performed'],
    'test_type': report['test_type'],
    'selected_by_group': report['selection']['selected_by_group'],
    'metrics': report['metrics'],
    'hard_positive_comparison': [
        {
            'filename': row['sample']['filename'],
            'full_prediction': row['full_prediction'],
            'full_probability': row['full_pv_probability'],
            'crop_prediction': row['primary_prediction'],
            'crop_probability': row['primary_pv_probability'],
            'crop_reason': row['views'][-1]['semantic']['reason'],
        }
        for row in report['results']
        if row['selection_group'] == 'hard_positive'
    ],
    'errors': [
        {
            'filename': row['sample']['filename'],
            'group': row['selection_group'],
            'full_prediction': row['full_prediction'],
            'primary_prediction': row['primary_prediction'],
        }
        for row in report['results']
        if not row['primary_correct']
    ],
    'runtime': report['runtime'],
    'remaining_issues': [
        '指标仍是图像级光伏存在性，不是像素级分割指标。',
        '局部裁剪由真实掩膜引导，只能用于训练期Teacher知识提取。',
        '本次不训练Student，结果用于决定是否开始蒸馏。',
    ],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Full report:', report_path)